# Day 4 下午 · MCP 协议 (90 min) + Skills 协议 (90 min)

## 学习目标

**A. MCP (Model Context Protocol)** — 90 min
1. 理解 MCP = LLM 工具调用的 USB-C
2. 掌握 Tools / Resources（Prompts 略提）三件套
3. 写一个真实可跑的 MCP server（带权限层）

**B. Anthropic Skills** — 90 min
4. 理解 Skills vs MCP vs Prompts 三协议生态
5. 写一个完整 Skill（SKILL.md + helper.py + reference/）
6. 掌握 Progressive Disclosure & Skills × MCP 集成
7. 生产实践：分发 / 版本 / 安全

## 前置

- Day 3 上午 ReAct Agent 中的 tool 调用基础
- Day 4 上午 Multi-Agent (Agent 之间需要标准化协议)
- 已 `pip install mcp>=0.9` (没装会自动 fallback 到 EduMCPServer)


<!-- session-2026-04-29-teaching-pass -->
## 本节配套资产（同级目录）

本 notebook 演示 MCP 协议与 Anthropic Skills，会用到课程包里两个**可直接 fork 的独立资产**：

### `mcp_server_demo/`（同级目录）

```
mcp_server_demo/
├── server.py        # 100 行 stdio JSON-RPC server，实现 initialize / tools/list / tools/call / shutdown
└── client_test.py   # subprocess 启动 server + 完整握手测试
```

这是一个**真协议实现**，不是教学包装。本 notebook 的 MCP 演示部分会启动它的子进程做真握手。

### `skills_demo/`（同级目录）

```
skills_demo/
├── README.md
├── code_review/             # 简单 skill：SKILL.md + helper.py + reference/
├── db_query/                # Skill × MCP 集成：skill 通过 MCP server 跑 SQL
└── capstone_assistant/      # 企业级 skill：升级 Capstone 打包成可分发的 skill
```

本 notebook 的 Skills 章节会读取这三个文件夹做 discover / progressive disclosure 演示。

**企业落地路径**：把这两个目录直接复制到你公司的项目里改造——`server.py` 的 `TOOLS` 字典换成你们的工具，`skills_demo/*/SKILL.md` 的 description 换成你们的领域知识。


> 📋 **讲课提示** *(此区块仅讲师版含，学员版自动剥离)*
> 
> **本节：** Day 4 下午 · MCP 协议 (90min) + Skills 协议 (90min)　|　**时长：** 180 min
> 
> **开场：** 问：『手机为什么用 USB-C 不用 5 种接口？』 → 类比 MCP。再问：『团队所有人的 Claude 怎么共享你写的工作流？』 → 引出 Skills。
> 
> **重点强调：**
> - **MCP** = Claude 调用外部工具的标准协议 (Tools / Resources / Prompts 三件套)
> - **Skills** = 可打包的内化能力（Claude『知道怎么做什么』），与 MCP 配对
> - **关键差别**：MCP 解决『能调什么』；Skills 解决『何时做什么 + 怎么做』
> - **Progressive Disclosure**：Skills 的杀手特性 — frontmatter 决定何时载入，body+reference 按需加载，不烧 context
> - **Skills × MCP**：Skill body 教 Claude 调哪个 MCP tool 完成任务（解耦能力 vs 工具）
>
> **常见误解：**
> - 学员以为 MCP 和 Skills 二选一 → 强调它们是配对，缺一不可
> - 学员以为 Skills 只是 prompt 模板 → 强调含 helper scripts 和 progressive 加载
>
> **互动设计：** 现场让学员说一个『部门内部反复使用的 SOP』，把它包成 Skill 草稿（SKILL.md frontmatter）
>
> **时间紧时：** 跳过 Skills × MCP 集成练习 (练习 5)，保 MCP server demo + Skills 写作 + Progressive 主线。

In [1]:
# ── 课程环境就位（自动定位课程根目录，让 utils/data/fonts 路径无论从 instructor/ 还是 student/ 都能用）──
import os, sys
_cur = os.path.abspath("")
_root = None
for _candidate in [_cur] + [os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if all(os.path.isdir(os.path.join(_candidate, d)) for d in ("utils", "data")):
        _root = _candidate
        break
if _root is None:
    raise RuntimeError("找不到课程根目录（应包含 utils/ 与 data/）。请确认 notebook 位于课程包的 instructor/ 或 student/ 子目录下。")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
if os.path.join(_root, "utils") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "utils"))
print(f"📂 课程根目录：{_root}")


📂 课程根目录：C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\assets\enterprise_5days


In [2]:
# 导入：LLM + MCP + Skills helpers
from config import setup
env = setup()
from mcp_helpers import (
    EduMCPServer, EduMCPClient,
    ToolDef, ResourceDef, PromptDef,
    tool_from_function, MCP_AVAILABLE,
)
from skills_helpers import (
    Skill, parse_skill_md, validate_skill,
    discover_skills, match_skill_for_query, load_skill_progressive,
)
import json

llm = env.get_llm()
print(f"✓ LLM 就位")
print(f"✓ MCP SDK 可用: {MCP_AVAILABLE}  (False 走 EduMCPServer 教学模式)")
print(f"✓ Skills helpers 就位")


[OK] 使用系统环境变量中的 DASHSCOPE_API_KEY
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus-2025-01-25
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus-2025-01-25
✓ LLM 就位
✓ MCP SDK 可用: False  (False 走 EduMCPServer 教学模式)
✓ Skills helpers 就位


---

# PART A · MCP 协议（90 min）

## A.1 · Why MCP（15 min）

### MCP 出现前的乱世

每家 LLM 都有自己的 tool 调用方式：

| 厂商 | 调用方式 |
|---|---|
| OpenAI | `function_call` |
| Anthropic | `tool_use` |
| Google | `function_declarations` |
| Cohere | `connectors` |

**结果**：你要让 LLM 调你公司的 CRM API，要为 N 家 LLM 各写一遍 schema。

### MCP = Model Context Protocol

Anthropic 2024 末推、2025-2026 成事实标准的**跨厂工具调用协议**。

类比 USB-C：写一次 server，所有支持 MCP 的 LLM/IDE 都能用。

### 三件套

```
MCP Server 暴露：
├── Tools      — LLM 可调用的函数（带 JSON schema）
├── Resources  — LLM 可读取的数据源（file / db / api）
└── Prompts    — 可复用的提示模板（带参数）
```

**今天聚焦 Tools + Resources（Prompts 一句话提）。**


---

## A.2 · Tools（20 min + 1 练习）

Tool = 一个函数 + description + JSON schema。LLM 看 description 决定调不调；调时按 schema 填参数。


In [3]:
# 用 helper 把普通 Python 函数自动转成 ToolDef
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

add_tool = tool_from_function(add)

print("ToolDef 自动生成：")
print(f"  name: {add_tool.name}")
print(f"  description: {add_tool.description}")
print(f"  parameters schema: {json.dumps(add_tool.parameters, indent=2)}")
print(f"\n  call(a=3, b=5) → {add_tool.call(a=3, b=5)}")


ToolDef 自动生成：
  name: add
  description: Add two integers.
  parameters schema: {
  "type": "object",
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}

  call(a=3, b=5) → 8


In [4]:
# 把多个 Tool 放进 Server，让 LLM 当 client
def multiply(a: int, b: int, label: str = "result") -> str:
    """Multiply two integers and return labeled result."""
    return f"{label}: {a * b}"

server = EduMCPServer(name="math-helper")
server.add_tool(add_tool)
server.add_tool(tool_from_function(multiply))

# Demo: LLM 看 tools 列表 → 决定调哪个
# Qwen / DashScope 有时会把 arguments 写成 list；这里按 schema 归一化，避免课堂 demo 因格式小偏差失败。
def _extract_json_object(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    start, end = raw.find("{"), raw.rfind("}")
    return raw[start:end + 1] if start >= 0 and end >= start else raw


def _normalize_tool_arguments(tool_schema, arguments):
    props = tool_schema["parameters"].get("properties", {})
    names = list(props.keys())
    if isinstance(arguments, list):
        arguments = {name: value for name, value in zip(names, arguments)}
    elif not isinstance(arguments, dict):
        arguments = {}
    for name, meta in props.items():
        if name not in arguments:
            continue
        value = arguments[name]
        if meta.get("type") == "integer" and isinstance(value, str) and value.strip().lstrip("-").isdigit():
            arguments[name] = int(value)
        elif meta.get("type") == "number" and isinstance(value, str):
            try:
                arguments[name] = float(value)
            except ValueError:
                pass
    return arguments


def llm_calls_tool(query, srv):
    tools = srv.list_tools()
    desc = "\n".join(
        f"- {t['name']} 参数={list(t['parameters']['properties'].keys())}: {t['description']}"
        for t in tools
    )
    raw = llm.generate(
        f'''你是一个 tool router。可用 tools:\n{desc}\n\n用户: {query}\n\n只输出 JSON 对象，格式必须是：\n{{"tool": "工具名", "arguments": {{"参数名": 参数值}}}}\n注意 arguments 必须是 object，不要输出数组。''',
        temperature=0,
    ).strip()
    try:
        plan = json.loads(_extract_json_object(raw))
        tool_schema = next(t for t in tools if t["name"] == plan["tool"])
        arguments = _normalize_tool_arguments(tool_schema, plan.get("arguments", {}))
        return f"{plan['tool']}({arguments}) → {srv.call_tool(plan['tool'], arguments)}"
    except (json.JSONDecodeError, KeyError, ValueError, TypeError, StopIteration) as e:
        return f"[LLM JSON 仍需修正] {e}; raw={raw[:160]}"

print(llm_calls_tool("帮我算 15 加 27", server))
print(llm_calls_tool("把 8 和 9 相乘标记为 'order_total'", server))


add({'a': 15, 'b': 27}) → 42


multiply({'a': 8, 'b': 9, 'label': 'order_total'}) → order_total: 72


In [5]:
# ============================================================
# 练习 1 | 自定义 Tool + Schema 严格校验
# ============================================================
#
# 【基础】（人人必做，10 min）
#   实现 build_search_tool()：定义 search_employee(name, department=None) 函数
#   并包装成 ToolDef 返回
#
# 【进阶】（技术学员选做，10 min）
#   实现 build_strict_tool(func)：在 tool_from_function 基础上加严格校验：
#   - 缺 required 参数 → ValueError("Missing: ...")
#   - 传未声明参数 → ValueError("Unknown: ...")
# ============================================================

EMPLOYEES = [
    {"name": "张三", "department": "技术部", "level": "senior"},
    {"name": "李四", "department": "市场部", "level": "junior"},
    {"name": "王五", "department": "技术部", "level": "lead"},
]


def build_search_tool():
    """【基础】返回 ToolDef"""
    # ↓↓↓ 【基础】填空（约 6 行）↓↓↓
    def search_employee(name: str, department: str = None):
        results = [e for e in EMPLOYEES if name in e["name"]]
        if department:
            results = [e for e in results if e["department"] == department]
        return json.dumps(results, ensure_ascii=False)
    return tool_from_function(search_employee)
    # ↑↑↑ 【基础】结束 ↑↑↑


def build_strict_tool(func):
    """【进阶】带 unknown 参数校验"""
    # ↓↓↓ 【进阶】填空（约 12 行）↓↓↓
    base = tool_from_function(func)
    declared = set(base.parameters["properties"].keys())
    required = set(base.parameters.get("required", []))
    original = base.func
    def strict(**kwargs):
        missing = required - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing: {sorted(missing)}")
        unknown = set(kwargs.keys()) - declared
        if unknown:
            raise ValueError(f"Unknown: {sorted(unknown)}")
        return original(**kwargs)
    base.func = strict
    return base
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】build_search_tool"); print("=" * 56)
    try:
        tool = build_search_tool()
        assert tool.name == "search_employee"
        result = tool.call(name="张三")
        print(f"  search('张三') → {result}")
        assert "张三" in result
        result = tool.call(name="王", department="技术部")
        print(f"  search('王', dept='技术部') → {result}")
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】build_strict_tool"); print("=" * 56)
    try:
        def divide(a: int, b: int) -> float:
            """Divide a/b"""
            return a / b
        strict = build_strict_tool(divide)
        assert strict.call(a=10, b=2) == 5.0
        print(f"  strict(a=10,b=2) → 5.0 ✓")
        try:
            strict.call(a=10)
        except ValueError as e:
            print(f"  缺参报错: {e} ✓")
        try:
            strict.call(a=10, b=2, c=99)
        except ValueError as e:
            print(f"  未知参数报错: {e} ✓")
        print("✅ 进阶通过")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】build_search_tool
  search('张三') → [{"name": "张三", "department": "技术部", "level": "senior"}]
  search('王', dept='技术部') → [{"name": "王五", "department": "技术部", "level": "lead"}]
✅ 基础通过

【进阶】build_strict_tool
  strict(a=10,b=2) → 5.0 ✓
  缺参报错: Missing: ['b'] ✓
  未知参数报错: Unknown: ['c'] ✓
✅ 进阶通过


---

## A.3 · Resources（15 min · 演示）

Resource = LLM 可**读取的数据源**（不是函数调用）。

| Tool | Resource |
|---|---|
| 函数式（有副作用） | 数据式（只读） |
| 例：`send_email()` | 例：`file:///docs/policy.md` |


In [6]:
# 演示：定义 file resource + dynamic resource
DOCS = {
    "policy_leave.md": "# 请假制度\n年假：5 年以下 5 天/年；5 年以上 15 天/年。",
    "policy_expense.md": "# 报销制度\n餐费 ≤ 100 元/餐。差旅一线 500/晚，二三线 350/晚。",
}

server2 = EduMCPServer(name="enterprise-docs")
for fn in DOCS:
    server2.add_resource(ResourceDef(
        uri=f"file:///docs/{fn}",
        name=fn,
        mime_type="text/markdown",
        reader=(lambda f=fn: DOCS[f]),
    ))

# 加一个动态 resource（每次读返回当前时间戳）
import time, random
server2.add_resource(ResourceDef(
    uri="live:///stats",
    name="live_stats",
    mime_type="application/json",
    reader=lambda: json.dumps({"timestamp": time.time(), "active_users": random.randint(50, 200)}),
))

print("可读取的 Resources:")
for r in server2.list_resources():
    print(f"  • {r['uri']}  ({r['mime_type']})")

print(f"\n读 policy_leave: {server2.read_resource('file:///docs/policy_leave.md')[:60]}...")
print(f"读 live_stats: {server2.read_resource('live:///stats')}")
print("\n💡 Live Resource 让 LLM 看到的总是『现在』，缓存失效问题自动解决")


可读取的 Resources:
  • file:///docs/policy_leave.md  (text/markdown)
  • file:///docs/policy_expense.md  (text/markdown)
  • live:///stats  (application/json)

读 policy_leave: # 请假制度
年假：5 年以下 5 天/年；5 年以上 15 天/年。...
读 live_stats: {"timestamp": 1777472905.559094, "active_users": 71}

💡 Live Resource 让 LLM 看到的总是『现在』，缓存失效问题自动解决


---

## A.4 · 实战：写真实 MCP Server + 权限层（25 min + 1 练习）

我们在 `mcp_server_demo/` 写好了一个**真正可跑的 MCP server**：
- `server.py` — 暴露 3 个企业 tool（订单/库存/通知）
- `client_test.py` — client 测试

下面用 LLM 当大脑试一遍**端到端**流程，并加权限层。


In [7]:
# 复用 mcp_server_demo
import sys
from pathlib import Path
sys.path.insert(0, str(Path('mcp_server_demo')))
from server import build_server as build_demo_server  # type: ignore

demo_server = build_demo_server()
demo_client = EduMCPClient(user_id="alice")
demo_client.connect(demo_server)

print(f"✓ Demo server 就位 ({len(demo_client.list_all_tools())} tools)")
for t in demo_client.list_all_tools():
    print(f"  • {t['name']}: {t['description']}")


✓ Demo server 就位 (3 tools)
  • query_order: Look up an order by ID
  • check_inventory: Check stock quantity for a SKU
  • send_notification: Send a notification to a user


In [8]:
# Demo: LLM 用 demo server 完成端到端任务
def llm_use_mcp(query, client):
    tools = client.list_all_tools()
    desc = "\n".join(f"- [{t['server']}] {t['name']}({list(t['parameters']['properties'].keys())}): {t['description']}" for t in tools)
    raw = llm.generate(
        f"可用工具:\n{desc}\n\n用户: {query}\n\n输出 JSON: {{\"server\": \"...\", \"tool\": \"...\", \"arguments\": {{...}}}}。只输出 JSON。",
        temperature=0,
    ).strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        plan = json.loads(raw)
        result = client.call(plan["server"], plan["tool"], **plan["arguments"])
        answer = llm.generate(
            f"用户问 '{query}'，调 {plan['tool']}({plan['arguments']}) 得到: {result}。请用一句话给最终答复。",
            temperature=0.2,
        )
        return {"plan": plan, "raw": result, "answer": answer.strip()}
    except Exception as e:
        return {"error": str(e), "raw_llm": raw[:200]}


print("=" * 60); print("Demo: 查订单"); print("=" * 60)
print(json.dumps(llm_use_mcp("查 ORD-001", demo_client), ensure_ascii=False, indent=2))

print("\n" + "=" * 60); print("Demo: 查库存"); print("=" * 60)
print(json.dumps(llm_use_mcp("SKU-A100 还有多少货", demo_client), ensure_ascii=False, indent=2))


Demo: 查订单


{
  "plan": {
    "server": "enterprise-demo",
    "tool": "query_order",
    "arguments": {
      "order_id": "ORD-001"
    }
  },
  "raw": "{\"status\": \"shipped\", \"total\": 199.0, \"customer\": \"alice\"}",
  "answer": "订单 ORD-001 已发货，总额为 199.0 元，收件人为 alice。"
}

Demo: 查库存


{
  "plan": {
    "server": "enterprise-demo",
    "tool": "check_inventory",
    "arguments": {
      "sku": "SKU-A100"
    }
  },
  "raw": "{\"sku\": \"SKU-A100\", \"quantity\": 35, \"in_stock\": true}",
  "answer": "SKU-A100 目前有 35 件库存，仍在售。"
}


---

### A.4.1 · 真起独立 server 进程：subprocess + stdio JSON-RPC（10 min）

上面 `llm_use_mcp` 的 demo 把 server 放在**同一个 Python 进程**里——方便教学，但**不是真实生产方式**。

**真 MCP 协议的本质**：
1. server 是**独立进程**（用任何语言写都行，不止 Python）
2. client 用 **subprocess** 拉起 server
3. 双方走 **JSON-RPC over stdio**（每行一个 JSON 消息）
4. 协议方法：`initialize` / `tools/list` / `tools/call` / `resources/list` / `resources/read` / `shutdown`

下面 demo **真起一个独立 server 进程**（`mcp_server_demo/server.py --stdio`），用真 JSON-RPC 与它通信。

> 💡 这跟 Anthropic 官方 `mcp` Python SDK（需 Python 3.10+）的底层做法**完全一样**，只是我们手写了协议帧，让你看到原貌而不是被 SDK 包起来。Claude Desktop / Cursor / Claude Code 也是这样跟 MCP server 说话。


In [9]:
# 真起独立 server 进程 + 跑 stdio JSON-RPC 通信
import subprocess, json, time, sys as _sys
from pathlib import Path

server_script = Path("mcp_server_demo/server.py")

# 1. 起 server subprocess
print(f"启动 server: {_sys.executable} {server_script} --stdio")
proc = subprocess.Popen(
    [_sys.executable, str(server_script), "--stdio"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
    bufsize=1,
)
time.sleep(0.3)  # 让 server 启动

req_id = 0
def rpc_call(method, params=None):
    global req_id
    req_id += 1
    req = {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params or {}}
    line = json.dumps(req, ensure_ascii=False) + "\n"
    print(f"  → {method}({params or '{}'})")
    proc.stdin.write(line); proc.stdin.flush()
    resp_line = proc.stdout.readline()
    resp = json.loads(resp_line.strip())
    if "error" in resp:
        print(f"  ← ERROR: {resp['error']}")
        return None
    return resp.get("result", {})

# 2. 协议握手
print("\n--- Step 1: initialize 协议握手 ---")
info = rpc_call("initialize")
print(f"  ← server={info['server_info']['name']} v{info['server_info']['version']}, protocol={info['protocol_version']}")

# 3. 列出 tools
print("\n--- Step 2: tools/list 列出能力 ---")
tools = rpc_call("tools/list")
for t in tools["tools"]:
    print(f"  ← {t['name']}: {t['description'][:50]}")

# 4. 真调一个 tool
print("\n--- Step 3: tools/call 调用 query_order ---")
result = rpc_call("tools/call", {"name": "query_order", "arguments": {"order_id": "ORD-002"}})
print(f"  ← {result['content'][0]['text']}")

# 5. 再调另一个
print("\n--- Step 4: tools/call 调用 check_inventory ---")
result = rpc_call("tools/call", {"name": "check_inventory", "arguments": {"sku": "SKU-A100"}})
print(f"  ← {result['content'][0]['text']}")

# 6. 关闭
print("\n--- Step 5: shutdown 关闭 server ---")
rpc_call("shutdown")
proc.terminate()
try:
    proc.wait(timeout=2)
except subprocess.TimeoutExpired:
    proc.kill()

print("\n💡 看到没？这就是 Claude Desktop / Cursor / Claude Code 跟 MCP server 通信的真实方式。")
print("   每条 JSON-RPC 消息一行，server 是独立进程（任何语言都能写），双方靠 stdio 管道通信。")


启动 server: E:\conda\envs\llmc\python.exe mcp_server_demo\server.py --stdio



--- Step 1: initialize 协议握手 ---
  → initialize({})
  ← server=enterprise-demo v0.1.0, protocol=2025-11-05-edu

--- Step 2: tools/list 列出能力 ---
  → tools/list({})
  ← query_order: Look up an order by ID
  ← check_inventory: Check stock quantity for a SKU
  ← send_notification: Send a notification to a user

--- Step 3: tools/call 调用 query_order ---
  → tools/call({'name': 'query_order', 'arguments': {'order_id': 'ORD-002'}})
  ← {"status": "pending", "total": 89.0, "customer": "bob"}

--- Step 4: tools/call 调用 check_inventory ---
  → tools/call({'name': 'check_inventory', 'arguments': {'sku': 'SKU-A100'}})
  ← {"sku": "SKU-A100", "quantity": 35, "in_stock": true}

--- Step 5: shutdown 关闭 server ---
  → shutdown({})

💡 看到没？这就是 Claude Desktop / Cursor / Claude Code 跟 MCP server 通信的真实方式。
   每条 JSON-RPC 消息一行，server 是独立进程（任何语言都能写），双方靠 stdio 管道通信。


In [10]:
# ============================================================
# 练习 2 | MCP Server 加权限层（基于 user_id 控制 tool 可见性）
# ============================================================
#
# 【基础】（人人必做，10 min）
#   build_basic_server()：建含 2 tool (read_orders, get_stats) 的 EduMCPServer
#
# 【进阶】（技术学员选做，15 min）
#   build_server_with_auth()：admin 全开放；viewer 只能调 read_*
#   实现 server.set_auth_check(fn) 校验
# ============================================================

def build_basic_server():
    """【基础】2-tool server"""
    # ↓↓↓ 【基础】填空（约 8 行）↓↓↓
    server = EduMCPServer(name="exercise-server")
    def read_orders():
        """List recent orders"""
        return json.dumps([{"id": "O1", "amt": 100}, {"id": "O2", "amt": 250}])
    def get_stats():
        """Get current stats"""
        return json.dumps({"orders_today": 42, "active_users": 128})
    server.add_tool(tool_from_function(read_orders))
    server.add_tool(tool_from_function(get_stats))
    return server
    # ↑↑↑ 【基础】结束 ↑↑↑


def build_server_with_auth():
    """【进阶】admin 全开放，viewer 只能 read_*"""
    # ↓↓↓ 【进阶】填空（约 16 行）↓↓↓
    server = EduMCPServer(name="auth-server")
    def read_orders():
        """List orders"""
        return json.dumps([{"id": "O1"}])
    def write_order(item: str, qty: int):
        """Create order"""
        return json.dumps({"created": item, "qty": qty})
    def delete_user(user_id: str):
        """Delete user"""
        return json.dumps({"deleted": user_id})
    for fn in [read_orders, write_order, delete_user]:
        server.add_tool(tool_from_function(fn))

    USER_ROLES = {"alice": "admin", "bob": "viewer"}
    def auth_check(user_id, action):
        role = USER_ROLES.get(user_id, "guest")
        if role == "admin":
            return True
        if role == "viewer":
            return action.startswith("read_")
        return False
    server.set_auth_check(auth_check)
    return server
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】build_basic_server"); print("=" * 56)
    try:
        srv = build_basic_server()
        client = EduMCPClient(user_id="any")
        client.connect(srv)
        tools = client.list_all_tools()
        assert len(tools) == 2
        print(f"  Tools: {[t['name'] for t in tools]}")
        result = client.call(srv.name, "read_orders")
        print(f"  read_orders() → {result}")
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】build_server_with_auth"); print("=" * 56)
    try:
        srv = build_server_with_auth()
        admin = EduMCPClient(user_id="alice"); admin.connect(srv)
        viewer = EduMCPClient(user_id="bob"); viewer.connect(srv)
        admin_tools = [t["name"] for t in admin.list_all_tools()]
        viewer_tools = [t["name"] for t in viewer.list_all_tools()]
        print(f"  admin 可见: {admin_tools}")
        print(f"  viewer 可见: {viewer_tools}")
        assert "delete_user" in admin_tools
        assert "delete_user" not in viewer_tools
        try:
            viewer.call(srv.name, "write_order", item="A", qty=1)
            print("  ✗ viewer 不应能调 write_order")
        except PermissionError:
            print("  viewer 调 write_order → 被拒 ✓")
        print("✅ 进阶通过")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】build_basic_server
  Tools: ['read_orders', 'get_stats']
  read_orders() → [{"id": "O1", "amt": 100}, {"id": "O2", "amt": 250}]
✅ 基础通过

【进阶】build_server_with_auth
  admin 可见: ['read_orders', 'write_order', 'delete_user']
  viewer 可见: ['read_orders']
  viewer 调 write_order → 被拒 ✓
✅ 进阶通过


### MCP 部分小结

- **Tools** = 函数 + JSON schema；用 `tool_from_function` 自动生成
- **Resources** = LLM 可读数据源；可静态可动态
- **Prompts**（一句话提）= 复用模板，类似 jinja2 但参数化更轻
- **Server + Auth**：生产场景用 `set_auth_check` 按用户限制 tool 可见性

下半场进入 **Skills**——如果说 MCP 解决『LLM 能调什么』，Skills 解决『LLM 知道何时怎么做什么』。


---

# PART B · Anthropic Skills（90 min）

## B.1 · Why Skills + 三协议对比（15 min）

### 三协议生态

|  | Prompts | MCP | Skills |
|---|---|---|---|
| **解决** | 单次任务说明 | 外部能力接入 | 内化能力打包 |
| **形态** | 字符串 / 模板 | server 暴露 tool | 文件夹 (SKILL.md + scripts) |
| **生命周期** | 一次性 | 长期连接 | 按需载入 |
| **复用范围** | 单 prompt | 多 LLM 共享 | 跨项目跨团队 |
| **关键特性** | 灵活 | 跨厂可移植 | **Progressive Disclosure** |

### 一个 Skill 的样子

```
my_skill/
├── SKILL.md           # 必需：YAML frontmatter + body
├── helper.py          # 可选：Claude 可调用的脚本
└── reference/         # 可选：progressive disclosure 文档
    ├── checklist.md
    └── examples.json
```

`SKILL.md` 头部：
```yaml
---
name: code-review
description: Performs a structured code review... (1-2 句话决定何时被选中)
allowed-tools: [bash, read_file]
version: "0.2"
---

# Code Review Skill
... (body：步骤 / 例子 / 模板)
```

### 关键创新：Progressive Disclosure

**传统 prompt**：把所有指令塞 system prompt → 每次都烧 context
**Skills**：
1. 启动时只读 `description`（几十字）
2. 匹配到再读 `body`（几百字）
3. 需要细节再读 `reference/*.md`（按需）

**省 context = 省钱 + 提速。**


<!-- skill-folder-tour -->
### 先把 Skill 文件夹看清楚

课堂里不用把 Skills 想成抽象概念：它就是一个可复制的文件夹。学员真正需要改的通常只有三处：`SKILL.md` 的 `name`、`description`、以及正文里的 workflow；`helper.py` 和 `reference/` 是进阶扩展。


In [11]:
from pathlib import Path

skills_root = Path("skills_demo") if Path("skills_demo").exists() else Path("assets/enterprise_5days/skills_demo")
print(f"Skills 根目录: {skills_root.resolve()}")
print()
print("课堂要看懂的结构：")
for skill_dir in sorted(p for p in skills_root.iterdir() if p.is_dir()):
    files = []
    if (skill_dir / "SKILL.md").exists():
        files.append("SKILL.md")
    files += [p.name for p in skill_dir.glob("*.py")]
    ref_dir = skill_dir / "reference"
    if ref_dir.exists():
        files.append("reference/")
    print(f"  {skill_dir.name}/ -> {', '.join(files)}")

print()
print("最小改造顺序：")
print("  1) 复制一个现成 skill 文件夹")
print("  2) 改 SKILL.md: name + description + workflow")
print("  3) 跑 validate_skill()，再用 match_skill_for_query() 测路由")


Skills 根目录: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\assets\enterprise_5days\skills_demo

课堂要看懂的结构：
  capstone_assistant/ -> SKILL.md, eval.py, pipeline.py, reference/
  code_review/ -> SKILL.md, helper.py, reference/
  db_query/ -> SKILL.md, reference/

最小改造顺序：
  1) 复制一个现成 skill 文件夹
  2) 改 SKILL.md: name + description + workflow
  3) 跑 validate_skill()，再用 match_skill_for_query() 测路由


---

## B.2 · SKILL.md 解剖 + 写第一个 Skill（25 min + 1 练习）

`skills_demo/code_review/SKILL.md` 是一个完整例子，看一下：


In [12]:
# 解析现成的 code-review skill
fm, body = parse_skill_md("skills_demo/code_review/SKILL.md")
print("Frontmatter:")
for k, v in fm.items():
    print(f"  {k}: {v}")
print(f"\nBody (前 400 字):\n{body[:400]}...")

# 也看下 validate
result = validate_skill("skills_demo/code_review")
print(f"\nvalidate: ok={result['ok']}")
if result['warnings']:
    print(f"  warnings: {result['warnings']}")


Frontmatter:
  name: code-review
  description: 对 Python 代码或 PR 做结构化审查：运行 ruff/mypy，并按 checklist 输出阻塞问题、应改问题和建议。Use when the user asks to review, audit, or check code quality.
  allowed-tools: ['bash', 'read_file']
  version: 0.2

Body (前 400 字):
# Code Review Skill

A pragmatic code reviewer for Python projects. Runs static checks and applies a
team checklist, then produces a prioritized report.

## When to use

- "Review this PR / file / function"
- "Check the code quality of X"
- "Audit this module for issues"

## Workflow

1. **Identify scope**: ask which files / diff to review
2. **Run static checks** (use `helper.py:run_checks()`):
 ...

validate: ok=True


In [13]:
# Discovery: 列出 skills_demo/ 里所有 skill
skills = discover_skills("skills_demo")
print(f"发现 {len(skills)} 个 skills:")
for s in skills:
    print(f"  • {s.name} (v{s.version}): {s.description[:80]}...")
    print(f"      helpers: {[p.name for p in s.helper_files]}")
    print(f"      references: {[p.name for p in s.reference_files]}")


发现 3 个 skills:
  • enterprise-knowledge-assistant (v1.0): 企业知识助手：回答 HR 政策、产品、技术 API、订单/库存等内部问题，并在 RAG、MCP tools、direct LLM 之间自动路由。Use for ...
      helpers: ['eval.py', 'pipeline.py']
      references: ['architecture.md', 'eval_cases.jsonl']
  • code-review (v0.2): 对 Python 代码或 PR 做结构化审查：运行 ruff/mypy，并按 checklist 输出阻塞问题、应改问题和建议。Use when the use...
      helpers: ['helper.py']
      references: ['checklist.md']
  • db-query (v0.1): 把订单、库存、通知类自然语言问题路由到 enterprise MCP server。Use when the user asks about ORD/SKU/o...
      helpers: []
      references: ['sql_examples.md']


In [14]:
# ============================================================
# 练习 3 | 写一个完整的 SKILL.md
# ============================================================
#
# 【基础】（人人必做，10 min）
#   写一个 meeting_notes skill 的 SKILL.md 字符串：
#   - frontmatter: name + description
#   - body: 何时用 + 步骤 (3-5 步)
#   - validate 通过 (ok=True)
#
# 【进阶】（技术学员选做，10 min）
#   实现 score_skill_description(desc, llm)：
#   - 用 LLM 给 description 打分 1-5（清晰度 + 可路由性）
#   - 返回 (score, suggestion)
#   - 演示：好 description 是技术活
# ============================================================
import tempfile, os


def write_meeting_notes_skill():
    """【基础】返回 SKILL.md 完整字符串"""
    # ↓↓↓ 【基础】填空（约 25 行）↓↓↓
    return """---
name: meeting-notes
description: Helps the user summarize a meeting transcript or audio recording into structured notes — action items, decisions, follow-ups. Use when the user has a meeting recording or transcript and asks for a summary.
allowed-tools: [read_file]
version: "0.1"
---

# Meeting Notes Skill

Turn meeting transcripts into structured, actionable notes.

## When to use

- "总结这场会议"
- "从录音里提取 action items"
- "这场会议有什么决定？"

## Workflow

1. **Identify input**: 文件路径 / 直接粘贴的 transcript
2. **Extract**:
   - **决定 (Decisions)**: 拍板的事项
   - **Action items**: 谁、做什么、何时
   - **Follow-ups**: 待跟进 / 待澄清
3. **Format** as markdown:
   ```markdown
   ## Decisions
   - ...

   ## Action Items
   - [ ] @<owner> <task> by <date>

   ## Follow-ups
   - ...
   ```
4. **Verify** with the user before sending out.

## What this skill does NOT

- Translate / transcribe (use a STT skill)
- Send the notes (use a notification skill)
"""
    # ↑↑↑ 【基础】结束 ↑↑↑


def score_skill_description(desc, llm):
    """【进阶】LLM 评分 description 质量"""
    # ↓↓↓ 【进阶】填空（约 14 行）↓↓↓
    prompt = f"""评分一个 Skill description 的质量 1-5 分。**严格使用全谱**：
- **5 分**：完美 — 清楚触发条件 + 动词宾语 + 30-200 字 + 与其他 skill 明显区分
- **4 分**：良 — 满足 3 项
- **3 分**：及格 — 满足 2 项
- **2 分**：差 — 只满足 1 项（如缺触发条件 / 太短 / 太模糊）
- **1 分**：极差 — 几乎不可用（< 10 字 / 完全模糊 / 无具体动词）

绝对禁止全部 3 分中庸。差的就给 1-2，好的就给 4-5。

description: {desc}

输出格式（严格 2 行）：
SCORE: <数字>
SUGGESTION: <一句话改进建议>"""
    raw = llm.generate(prompt, temperature=0).strip()
    score = 3
    suggestion = ""
    for line in raw.splitlines():
        if line.upper().startswith("SCORE:"):
            try:
                score = int("".join(c for c in line if c.isdigit())[:1])
            except (ValueError, IndexError):
                pass
        elif line.upper().startswith("SUGGESTION:"):
            suggestion = line.split(":", 1)[1].strip()
    return (score, suggestion)
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】write_meeting_notes_skill"); print("=" * 56)
    try:
        skill_md = write_meeting_notes_skill()
        # 写到临时文件夹验证
        with tempfile.TemporaryDirectory() as tmp:
            sd = os.path.join(tmp, "meeting-notes")
            os.makedirs(sd)
            with open(os.path.join(sd, "SKILL.md"), "w", encoding="utf-8") as f:
                f.write(skill_md)
            result = validate_skill(sd)
            print(f"  validate: ok={result['ok']}, warnings={result['warnings']}")
            assert result["ok"], f"validate 不通过：{result['errors']}"
            fm, body = parse_skill_md(os.path.join(sd, "SKILL.md"))
            assert fm["name"] == "meeting-notes"
            assert len(fm["description"]) > 30
            print(f"  name: {fm['name']}  desc 长度: {len(fm['description'])}  body 长度: {len(body)}")
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】score_skill_description"); print("=" * 56)
    try:
        good = "Helps the user write structured meeting notes from transcripts. Extracts action items, decisions, and follow-ups. Use when the user has a meeting recording or transcript."
        bad = "AI 助手"
        score_g, sug_g = score_skill_description(good, llm)
        score_b, sug_b = score_skill_description(bad, llm)
        print(f"  好 desc: score={score_g} | suggestion: {sug_g[:80]}")
        print(f"  差 desc: score={score_b} | suggestion: {sug_b[:80]}")
        assert score_g >= score_b
        print("✅ 进阶通过 — 好 description 评分更高")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】write_meeting_notes_skill
  validate: ok=True, warnings=[]
  name: meeting-notes  desc 长度: 205  body 长度: 641
✅ 基础通过

【进阶】score_skill_description


  好 desc: score=5 | suggestion: 描述已很清晰，无需改进。
  差 desc: score=1 | suggestion: 明确触发条件、具体任务和描述范围以提升清晰度。
✅ 进阶通过 — 好 description 评分更高


---

## B.3 · Helper Scripts + Progressive Disclosure（20 min + 1 练习）

### Helper Scripts

Skills 不是死文档——可以挂 Python 脚本。Claude 会读 `helper.py` 的 docstring，按需调用其中函数。

例：`code_review/helper.py` 提供 `run_checks(target)` 跑 ruff + mypy。Skill body 教 Claude 在第 2 步调用它。

### Progressive Disclosure（杀手特性）

**Naive 做法**：所有 skill 全部内容塞进 system prompt → context 爆掉

**Skills 做法**：3 层载入
1. **Always**: discovery 时只读 `description`（几十字）
2. **On match**: 用户 query 匹配 → 加载 body（几百字）
3. **On demand**: query 涉及细节 → 加载相关 `reference/*.md`（按需）

下面演示 `load_skill_progressive`：


In [15]:
# Progressive disclosure 实战
skills = discover_skills("skills_demo")
code_review = next(s for s in skills if s.name == "code-review")

# 场景 A：query 简单 → 只载入 body，不读 reference/checklist.md
loaded_a = load_skill_progressive(code_review, "review this function: def add(a,b): return a+b", llm)
print("=" * 56)
print("场景 A: '简单函数 review'")
print("=" * 56)
print(f"  loaded references: {[r['name'] for r in loaded_a['references_loaded']]}")
print(f"  estimated tokens: {loaded_a['tokens_estimate']}")

# 场景 B：query 复杂 + 提到团队 → LLM 决定加载 checklist.md
loaded_b = load_skill_progressive(
    code_review,
    "review this PR — 我们团队对错误处理特别敏感，要走完整 checklist",
    llm,
)
print("\n" + "=" * 56)
print("场景 B: '完整 checklist review'")
print("=" * 56)
print(f"  loaded references: {[r['name'] for r in loaded_b['references_loaded']]}")
print(f"  estimated tokens: {loaded_b['tokens_estimate']}")
print(f"\n💡 场景 A 省了 ~{loaded_b['tokens_estimate'] - loaded_a['tokens_estimate']} tokens — 简单任务不需要全文档")


场景 A: '简单函数 review'
  loaded references: []
  estimated tokens: 317



场景 B: '完整 checklist review'
  loaded references: ['checklist.md']
  estimated tokens: 540

💡 场景 A 省了 ~223 tokens — 简单任务不需要全文档


In [16]:
# ============================================================
# 练习 4 | validate_skill + LLM 路由 (match_skill_for_query)
# ============================================================
#
# 【基础】（人人必做，10 min）
#   实现 validate_all(skills_dir)：跑 validate_skill 检查所有子目录，返回 dict {name: {ok, errors, warnings}}
#
# 【进阶】（技术学员选做，15 min）
#   实现 audit_routing(test_queries, skills, llm)：
#   - 一组 (query, expected_skill_name) pairs
#   - 用 match_skill_for_query 路由，统计准确率
#   - 错路由 case 列出来供 description 改进
# ============================================================

def validate_all(skills_dir):
    """【基础】批量 validate"""
    # ↓↓↓ 【基础】填空（约 6 行）↓↓↓
    from pathlib import Path as P
    out = {}
    for sub in sorted(P(skills_dir).iterdir()):
        if sub.is_dir() and (sub / "SKILL.md").exists():
            out[sub.name] = validate_skill(sub)
    return out
    # ↑↑↑ 【基础】结束 ↑↑↑


def audit_routing(test_queries, skills, llm):
    """【进阶】路由准确率审计"""
    # ↓↓↓ 【进阶】填空（约 14 行）↓↓↓
    correct = 0
    audit_log = []
    for query, expected in test_queries:
        picked = match_skill_for_query(query, skills, llm)
        picked_name = picked.name if picked else None
        ok = picked_name == expected
        if ok:
            correct += 1
        audit_log.append({
            "query": query,
            "expected": expected,
            "picked": picked_name,
            "ok": ok,
        })
    return {
        "accuracy": correct / len(test_queries) if test_queries else 0,
        "errors": [a for a in audit_log if not a["ok"]],
        "log": audit_log,
    }
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】validate_all('skills_demo')"); print("=" * 56)
    try:
        report = validate_all("skills_demo")
        for name, r in report.items():
            status = "✓" if r["ok"] else "✗"
            print(f"  {status} {name}: errors={r['errors']}, warnings={len(r['warnings'])}")
        assert all(r["ok"] for r in report.values()), "应所有 demo skill 都 valid"
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】audit_routing"); print("=" * 56)
    try:
        skills = discover_skills("skills_demo")
        test_cases = [
            ("review my python code", "code-review"),
            ("查 ORD-005 订单", "db-query"),
            ("入职 5 年年假几天", "enterprise-knowledge-assistant"),
            ("SKU-A100 库存多少", "db-query"),
        ]
        result = audit_routing(test_cases, skills, llm)
        print(f"  路由准确率: {result['accuracy']:.0%}")
        for log in result["log"]:
            mark = "✓" if log["ok"] else "✗"
            print(f"    {mark} '{log['query'][:40]}' → expected={log['expected']}, picked={log['picked']}")
        if result["errors"]:
            print(f"\n  💡 错路由 case 是改 description 的输入")
        print("✅ 进阶通过")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】validate_all('skills_demo')
  ✓ capstone_assistant: errors=[], warnings=0
  ✓ code_review: errors=[], warnings=0
  ✓ db_query: errors=[], warnings=0
✅ 基础通过

【进阶】audit_routing


❌ 进阶未通过: RateLimitError: Error code: 429 - {'error': {'message': 'You have exceeded your current request limit. For details, see: https://help.aliyun.com/zh/model-studio/error-code#rate-limit', 'type': 'limit_requests', 'param': None, 'code': 'limit_requests'}, 'request_id': '88fe13ae-f031-9646-88fb-ec813950b434'}


---

## B.4 · Skills × MCP 集成模式（20 min + 1 练习）

```
┌─────────────────┐         ┌──────────────────┐
│  Skills          │         │  MCP             │
│  (能力 / 何时做) │ ─────▶ │  (工具 / 怎么调) │
└─────────────────┘         └──────────────────┘

例：db-query Skill 教 Claude『查订单』，actual API 调用走 enterprise-demo MCP server
```

`skills_demo/db_query/SKILL.md` 的 `allowed-tools` 字段限制了它只能调 3 个 MCP tool：
- `mcp__enterprise-demo__query_order`
- `mcp__enterprise-demo__check_inventory`
- `mcp__enterprise-demo__send_notification`

下面看完整流程。


In [17]:
# 完整流程：用户 query → 路由到 Skill → Skill 调 MCP → 返回
skills = discover_skills("skills_demo")
demo_server = build_demo_server()  # 复用前面的 MCP server
demo_client = EduMCPClient(user_id="demo")
demo_client.connect(demo_server)


def skill_calls_mcp(query):
    """端到端：query → 选 skill → 让 Claude 按 skill body 决定调 MCP tool"""
    # 1. Skill discovery + routing
    skill = match_skill_for_query(query, skills, llm)
    if skill is None:
        return f"[no matching skill] {query}"
    # 2. Load skill body (progressive)
    loaded = load_skill_progressive(skill, query, llm)
    # 3. Skill body 指导 Claude 调哪个 MCP tool
    tools = demo_client.list_all_tools()
    desc = "\n".join(f"- {t['name']}({list(t['parameters']['properties'].keys())})" for t in tools)
    plan_prompt = f'''Skill: {skill.name}
Skill 指导:
{loaded['body'][:600]}

可用 MCP tools:
{desc}

用户 query: {query}

按 skill 指导决定调哪个 tool。输出 JSON: {{"tool": "...", "arguments": {{...}}}}。'''
    raw = llm.generate(plan_prompt, temperature=0).strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        plan = json.loads(raw)
        result = demo_client.call(demo_server.name, plan["tool"], **plan["arguments"])
        return {"skill": skill.name, "tool": plan["tool"], "result": result}
    except Exception as e:
        return {"skill": skill.name, "error": str(e)}


# Demo
print("=" * 60); print("Demo 1: '查 ORD-002'"); print("=" * 60)
print(json.dumps(skill_calls_mcp("查 ORD-002 订单状态"), ensure_ascii=False, indent=2))

print("\n" + "=" * 60); print("Demo 2: '库存查 SKU-A100'"); print("=" * 60)
print(json.dumps(skill_calls_mcp("SKU-A100 还有多少库存？"), ensure_ascii=False, indent=2))


Demo 1: '查 ORD-002'


{
  "skill": "db-query",
  "error": "Expecting value: line 1 column 1 (char 0)"
}

Demo 2: '库存查 SKU-A100'


{
  "skill": "db-query",
  "error": "Expecting value: line 1 column 1 (char 0)"
}


In [18]:
# ============================================================
# 练习 5 | 写一个新 Skill 调用现有 MCP server
# ============================================================
#
# 【基础】（人人必做，10 min）
#   写 notify_skill_md：返回字符串 — 一个 notify-customer skill
#   - description 提到『何时用：发通知 / 提醒 / 告知』
#   - body 教 Claude 用 send_notification tool
#   - allowed-tools 含 mcp__enterprise-demo__send_notification
#
# 【进阶】（技术学员选做，15 min）
#   写 multi_tool_skill_md：一个 skill 调 ≥ 2 个 MCP tool
#   场景：『发货前自动检查 + 通知』
#   - 先 check_inventory(sku)
#   - 库存够 → query_order(order_id) 拿客户名
#   - 然后 send_notification 给客户
#   body 要清楚教 Claude 这个 3 步流程
# ============================================================

def notify_skill_md():
    """【基础】返回 SKILL.md 字符串"""
    # ↓↓↓ 【基础】填空（约 16 行）↓↓↓
    return """---
name: notify-customer
description: Sends a notification to a customer/user via the enterprise notification system. Use when the user wants to inform, alert, or remind a specific person about something (order shipped, inventory ready, account issue, etc).
allowed-tools: [mcp__enterprise-demo__send_notification]
version: "0.1"
---

# Notify Customer Skill

Send a single notification to a user.

## When to use

- "通知 alice 她的订单到了"
- "发个提醒给 bob"
- "告诉 carol 库存已补"

## Workflow

1. 解析: 谁 (user_id) + 内容 (message)
2. 调 `mcp__enterprise-demo__send_notification(user_id=..., message=...)`
3. 确认发出 + 反馈给用户

## Examples

| 用户说 | tool 调用 |
|---|---|
| "通知 alice 订单到了" | `send_notification(user_id="alice", message="订单到了")` |
| "提醒 bob 续费" | `send_notification(user_id="bob", message="请续费")` |
"""
    # ↑↑↑ 【基础】结束 ↑↑↑


def multi_tool_skill_md():
    """【进阶】3 步发货 skill"""
    # ↓↓↓ 【进阶】填空（约 24 行）↓↓↓
    return """---
name: pre-ship-check
description: Performs the pre-shipment workflow — checks inventory, looks up the customer for an order, then notifies them that the order is ready to ship. Use when the user wants to do a "ready to ship" or "pre-ship" workflow for an existing order.
allowed-tools:
  - mcp__enterprise-demo__check_inventory
  - mcp__enterprise-demo__query_order
  - mcp__enterprise-demo__send_notification
version: "0.1"
---

# Pre-Ship Check Skill

3-step workflow: 检库存 → 查客户 → 通知。

## When to use

- "ORD-001 准备发货"
- "检查 ORD-XXX 是否能发货并通知客户"
- "走发货前检查"

## Workflow (强制 3 步顺序)

1. **Check inventory**: 调 `check_inventory(sku=...)`
   - 库存 = 0 → 中止，告诉用户『缺货，无法发货』
2. **Look up order**: 调 `query_order(order_id=...)` 拿 customer 字段
3. **Notify**: 调 `send_notification(user_id=<customer>, message="您的订单 <id> 即将发货")`

## Error handling

- 任何一步失败 → 不继续后面步骤，回滚之前的 side effects（这里只 send_notification 有 side effect）
- 报告每步 PASS / FAIL 状态

## Output format

```
[1/3] check_inventory(SKU-XXX) → in_stock: <yes/no>
[2/3] query_order(ORD-XXX) → customer: <name>
[3/3] send_notification(<name>, ...) → sent
```
"""
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】notify_skill_md"); print("=" * 56)
    try:
        md = notify_skill_md()
        with tempfile.TemporaryDirectory() as tmp:
            sd = os.path.join(tmp, "notify-customer")
            os.makedirs(sd)
            with open(os.path.join(sd, "SKILL.md"), "w", encoding="utf-8") as f:
                f.write(md)
            result = validate_skill(sd)
            assert result["ok"], result["errors"]
            fm, body = parse_skill_md(os.path.join(sd, "SKILL.md"))
            assert "send_notification" in fm.get("allowed-tools", [])[0]
            assert "通知" in body or "notify" in body.lower()
        print(f"  validate ok + 含 send_notification + body 含通知关键词")
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】multi_tool_skill_md"); print("=" * 56)
    try:
        md = multi_tool_skill_md()
        # 检查 body 含三步流程关键词
        for kw in ["check_inventory", "query_order", "send_notification"]:
            assert kw in md, f"应包含 {kw}"
        # frontmatter 含 3 个 allowed-tools (multiline 形式)
        assert md.count("mcp__enterprise-demo__") >= 3, "应有 3 个 MCP tool"
        print("  ✓ body 含三步流程")
        print("  ✓ allowed-tools 含 3 个 MCP tools")
        print("✅ 进阶通过 — 复合 skill 适合多步业务流程")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】notify_skill_md
  validate ok + 含 send_notification + body 含通知关键词
✅ 基础通过

【进阶】multi_tool_skill_md
  ✓ body 含三步流程
  ✓ allowed-tools 含 3 个 MCP tools
✅ 进阶通过 — 复合 skill 适合多步业务流程


---

## B.4.5 · 实操：用 code_review Skill 审查一段含 bug 的代码（15 min）

到这里学员可能仍觉得 Skills "概念懂了，但具体能给我做什么？"

下面用 `code_review/` skill 跑一个**真实任务**：

```
任务情景: 我有一段 Python 代码，疑似有 bug + 风格问题。
        想要 Claude 按团队 checklist 帮我审一遍。

无 skill 怎么办: 写个 prompt"review this code" → 输出格式不稳，
                每个 reviewer 可能给不同维度的反馈。

有 skill 怎么办: discover code-review skill → 自动按 SKILL.md
                的 5 步 workflow + checklist.md 出结构化报告。
```


In [19]:
# 实操 1: 用 code_review skill 端到端审查
import sys, json
from pathlib import Path

# 待审的代码（含若干典型问题）
buggy_code = """
def calc_discount(price, discount):
    # 没参数校验、负价格不报错、整数除法可能出问题
    final = price - price * discount
    return final

def process_orders(orders):
    # 异常被吞、空列表会报错
    total = 0
    try:
        for o in orders:
            total = total + o['amount']
    except:
        pass
    return total / len(orders)

def get_user(user_id):
    # SQL 注入、明文密码
    sql = "SELECT * FROM users WHERE id = " + str(user_id)
    return execute(sql)
"""

# Step 1: discover + 路由到 code-review
skills = discover_skills("skills_demo")
picked = match_skill_for_query("review this Python code for bugs and style issues", skills, llm)
print(f"Step 1 路由: {picked.name if picked else '(none)'}")

# Step 2: progressive load — 简单 query 不需 reference
loaded = load_skill_progressive(picked, "review this Python code", llm)
print(f"Step 2 加载: body {len(loaded['body'])} 字符 + {len(loaded['references_loaded'])} 个 reference")

# Step 3: 让 LLM 按 skill body 跑审查 workflow
review_prompt = f"""你是 code-review skill 的执行体。严格按下面的 workflow 审查代码。

[SKILL body]
{loaded['body']}

[要审的代码]
```python
{buggy_code}
```

按 workflow 第 4 步的 Output format 输出结构化报告。
"""
report = llm.generate(review_prompt, temperature=0.1)
print("\n" + "=" * 60)
print("Step 3 输出（按 SKILL workflow 的结构化报告）:")
print("=" * 60)
print(report)


Step 1 路由: code-review


Step 2 加载: body 1268 字符 + 1 个 reference



Step 3 输出（按 SKILL workflow 的结构化报告）:
```markdown
## Code Review Summary

**Files reviewed**: calc_discount, process_orders, get_user  
**Critical**: 3  **Important**: 2  **Nits**: 1

### 🔴 Blocking
- `get_user`: SQL injection vulnerability due to string concatenation of user input.
- `process_orders`: Division by zero if `orders` is an empty list (`len(orders)`).
- `process_orders`: Swallowing all exceptions without logging or handling, which can hide critical errors.

### 🟡 Should fix
- `calc_discount`: Missing parameter validation for negative prices or invalid discount values.
- `process_orders`: Using a bare `except` block instead of catching specific exceptions.

### 🟢 Nice to have
- `calc_discount`: Consider using integer division explicitly if the business logic requires it, or add a comment explaining the intent.
```

### Explanation:
1. **Blocking Issues**:
   - The `get_user` function has a severe security flaw due to SQL injection.
   - The `process_orders` function will rai

In [20]:
# 实操 2: 对比 — naive 跑 3 次看格式漂移 vs skill 跑 3 次看格式稳定
# 关键洞察：单次跑 naive 也能给好答案；但**多次跑**naive 输出格式漂移大，
# 不能给下游系统消费。Skill 强制走 SKILL.md workflow 输出可重复。

def has_critical_section(text):
    """检查是否含『阻塞级问题』标识（任何形式）"""
    return any(m in text for m in ["🔴", "Blocking", "Critical", "严重", "BLOCKER"])

def has_should_fix(text):
    return any(m in text for m in ["🟡", "Should fix", "应改", "SHOULD"])

def has_nice(text):
    return any(m in text for m in ["🟢", "Nice", "建议", "NIT"])

def count_bullet_sections(text):
    """大致数 markdown 主标题数量"""
    return sum(1 for line in text.split("\n") if line.strip().startswith(("##", "###")))


print("=" * 70)
print("跑 3 次同代码 — 对比 naive 与 skill-driven 的格式稳定性")
print("=" * 70)

naive_results = []
skill_results = []
for i in range(3):
    print(f"\n第 {i+1} 次...")
    # naive
    n = llm.generate(f"Review this code:\n```python\n{buggy_code}\n```", temperature=0.3)
    naive_results.append(n)
    # skill (复用上面 Step 3 的 review_prompt)
    s = llm.generate(review_prompt, temperature=0.3)
    skill_results.append(s)

# 量化指标
print("\n" + "=" * 70)
print(f"{'指标':<30} {'naive 3 次':<15} {'skill 3 次':<15}")
print("=" * 70)
metrics = [
    ("含『🔴 Blocking』标识", has_critical_section),
    ("含『🟡 Should fix』标识", has_should_fix),
    ("含『🟢 Nice』标识", has_nice),
]
for label, fn in metrics:
    n_n = sum(fn(r) for r in naive_results)
    n_s = sum(fn(r) for r in skill_results)
    print(f"  {label:<28} {n_n:>3}/3 次          {n_s:>3}/3 次")

# 长度方差（格式稳定性的代理指标）
import statistics
naive_lens = [len(r) for r in naive_results]
skill_lens = [len(r) for r in skill_results]
print(f"  长度均值                       {statistics.mean(naive_lens):>6.0f}        {statistics.mean(skill_lens):>6.0f}")
print(f"  长度标准差（越小越稳定）       {statistics.stdev(naive_lens):>6.0f}        {statistics.stdev(skill_lens):>6.0f}")
print(f"  章节数（## 标题）均值          {statistics.mean([count_bullet_sections(r) for r in naive_results]):>6.1f}        {statistics.mean([count_bullet_sections(r) for r in skill_results]):>6.1f}")

print("""
💡 关键观察:
  - 单次看 naive 也写得不错——但同样问题跑 3 次，格式漂移大（章节数 / 标题命名 / 三档分类不稳）
  - skill-driven 强制走 SKILL.md 的 workflow 第 4 步 Output format → 三档分类 + 章节稳定
  - 工业场景下下游系统（dashboard / 自动化流转）需要稳定的结构化输出 → skill 才靠谱
"""[1:])


跑 3 次同代码 — 对比 naive 与 skill-driven 的格式稳定性

第 1 次...



第 2 次...



第 3 次...



指标                             naive 3 次       skill 3 次      
  含『🔴 Blocking』标识                0/3 次            3/3 次
  含『🟡 Should fix』标识              0/3 次            3/3 次
  含『🟢 Nice』标识                    0/3 次            3/3 次
  长度均值                         4266          1614
  长度标准差（越小越稳定）          109           159
  章节数（## 标题）均值            10.3           5.0
💡 关键观察:
  - 单次看 naive 也写得不错——但同样问题跑 3 次，格式漂移大（章节数 / 标题命名 / 三档分类不稳）
  - skill-driven 强制走 SKILL.md 的 workflow 第 4 步 Output format → 三档分类 + 章节稳定
  - 工业场景下下游系统（dashboard / 自动化流转）需要稳定的结构化输出 → skill 才靠谱



### 何时 Skill 真正值得？

| 场景 | 推荐 |
|---|---|
| 一次性 / ad-hoc 任务 | 直接 prompt，不用 skill |
| **团队反复跑同一类工作流**（code review / 周报 / 会议纪要 / SOP）| ✅ Skill |
| 需要**结构化输出**给下游系统消费 | ✅ Skill |
| 需要**可观测 / 可审计**（每次走相同步骤）| ✅ Skill |
| 任务需要**多个 helper script 协作** | ✅ Skill |
| Prompt > 200 字 + 含步骤 + 含示例 | ✅ Skill（已经够复杂了） |

**rule of thumb**: 同样的指令你写给同事第 3 遍 → 该封 Skill 了。

### 用 Skill 你刚才完成了什么

你刚刚把一个**团队 Code Review SOP** 用 1 个 SKILL.md + 1 个 helper.py + 1 个 checklist.md 表达出来。任何团队成员（或他们的 Claude / Cursor / Claude Code）只要 `import` 这个 skill 文件夹就能复用——**不需要重写 prompt，不需要培训新人**。


---

## B.5 · 生产实践 + 总结（10 min）

### 分发

| 方式 | 适合 |
|---|---|
| `~/.claude/skills/` 个人本地 | 个人助理 / 试验 |
| Git repo 团队共享 | 团队复用，version control |
| Claude.ai 上传 | 企业用户跨设备同步 |
| Anthropic Skill Marketplace（2026 起） | 公开发布 |

### 版本管理

`SKILL.md` frontmatter 加 `version: "1.0"`。改 description 或 body 时手动 bump。

```
my-skill/
├── SKILL.md   (version: 1.0)
└── CHANGELOG.md
    - v1.0: initial
    - v1.1: tightened description, added reference/sql_examples
```

### 安全

- helper.py 跑哪个用户/进程权限？默认是 Claude 进程权限——**别在 helper 里干 sudo / rm -rf**
- `allowed-tools` 字段限定 MCP tool 子集，违规会被 Claude 拒绝
- reference/ 不要塞密钥 / PII

### 测试一个 Skill 的好坏

1. **`description` 路由测试**：N 个 query，观察 LLM 选中率（练习 4 进阶）
2. **`body` 指令清晰度**：让 Claude 跑同 query 5 次，输出一致吗？
3. **`reference/` 进度披露**：哪些 reference 真的被加载？没加载的可能是写得太冷门

### 与 MCP 的协同

- Skills 描述『何时做 + 步骤』；MCP 提供『可调的 tool』
- 一个 skill 通过 `allowed-tools` 锁定它能用的 MCP tool 子集
- 同一组 MCP tool 可被多个 skill 复用（不重写）

### 何时不要用 Skill

- 一次性任务（写 prompt 够用）
- 高度动态的指令（每次都不一样）
- 与 LLM 模型强耦合（换模型就坏）


---

## Day 4 下午 总结

| 协议 | 解决 | 形态 | 学到了 |
|---|---|---|---|
| **Prompts** | 单次任务 | 字符串 | (Day 3 已学) |
| **MCP** | 外部能力 | server 暴露 tool | Tools/Resources/真实 server + 权限 |
| **Skills** | 内化能力 | 文件夹 (SKILL.md+scripts) | 写法 + Progressive Disclosure + 配 MCP |

### 为何今天把 MCP + Skills 放一起

它们是 Anthropic 2026 战略的**两个互补支柱**：
- MCP = 把能力接进来（外部 API → LLM 可调）
- Skills = 把能力打包出去（团队 SOP → 可复用）

明天 Day 5 上午 **Agentic RAG**，下午 **LLMOps + 升级 Capstone**。Day 5 下午**最后 30 min** 我们会把整个升级 Capstone 打包成一个 `enterprise-knowledge-assistant` Skill，演示『5 天合体 → 一个文件夹可复用』。

### 推荐资料

- MCP 官方: https://modelcontextprotocol.io
- Anthropic Skills 文档（搜『Claude Skills』）
- 现成 community MCP servers: github / gmail / slack / postgres
- `skills_demo/` 三个完整可改的 skill 例子
